In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

In [13]:
# Load data, combine and convert to ndarrays

fname = 'daily32_RAD_BSH_TMP_LAT.pickle'
with open(fname, 'rb') as f:
    data_dict = pickle.load(f)
    
print(data_dict['Aksaray'].head())

            Station  Insolation  Daily Total  Temperature(avg)  Latitude
Date                                                                    
2011-01-01  Aksaray         1.7      72927.6         -0.025000   38.3705
2011-01-02  Aksaray         5.6     145874.7          2.541667   38.3705
2011-01-03  Aksaray         6.3     147197.8          6.666667   38.3705
2011-01-04  Aksaray         2.9     100350.9          5.312500   38.3705
2011-01-05  Aksaray         0.7      42138.7          4.639130   38.3705


In [14]:
aks=data_dict['Aksaray']
adate = aks.index.values[424]
print(adate)
print(type(adate))

year = pd.DatetimeIndex([adate]).year[0]
month = pd.DatetimeIndex([adate]).month[0]
day = pd.DatetimeIndex([adate]).day[0]
print('Year {} - Month {} - Day {}'.format(year,month,day))
print('{} - {} - {}\n'.format(type(year), type(month), type(day)))

#year, month, day = int(year[0]), int(month[0]), int(day[0])
print('Year {} - Month {} - Day {}'.format(year,month,day))
print('{} - {} - {}\n'.format(type(year), type(month), type(day)))
print(aks.shape)
print(aks.head())
ak = aks.reset_index()

2012-02-29T00:00:00.000000000
<class 'numpy.datetime64'>
Year 2012 - Month 2 - Day 29
<class 'numpy.int64'> - <class 'numpy.int64'> - <class 'numpy.int64'>

Year 2012 - Month 2 - Day 29
<class 'numpy.int64'> - <class 'numpy.int64'> - <class 'numpy.int64'>

(1816, 5)
            Station  Insolation  Daily Total  Temperature(avg)  Latitude
Date                                                                    
2011-01-01  Aksaray         1.7      72927.6         -0.025000   38.3705
2011-01-02  Aksaray         5.6     145874.7          2.541667   38.3705
2011-01-03  Aksaray         6.3     147197.8          6.666667   38.3705
2011-01-04  Aksaray         2.9     100350.9          5.312500   38.3705
2011-01-05  Aksaray         0.7      42138.7          4.639130   38.3705


In [15]:
aks = data_dict['Aksaray']
adate = pd.DatetimeIndex(aks.index)[424]
print(adate)
print(type(adate))
def day_of_year(date):
    new_year_day = pd.Timestamp(year=date.year, month=1, day=1)
    return (date - new_year_day).days + 1
print(day_of_year(adate))

2012-02-29 00:00:00
<class 'pandas._libs.tslib.Timestamp'>
60


In [16]:
# We need to add H0 and maybe N (total daylight hours) to the dataset. They can be calculated.

def h0(row):
    # Get a row from a dataframe, give back H0 and daylength
    # Leap year must be taken into account
    
    # row['Latitude'] and row['Date'] are relevant inputs
    
    # phi is taken in degrees, all angles are assumed to be degrees as well in formulas
    # numpy defaults to radians however...
    
    gsc = 1367
    phi = np.deg2rad(row['Latitude'])
    date = row['Date']
    
    year = pd.DatetimeIndex([date]).year[0]
    day = day_of_year(date)
    
    if year % 4 == 0:
        B = (day-1) * (360/366)
    else:
        B = (day-1) * (360/365)
    
    B = np.deg2rad(B)
    delta = (180/np.pi) * (0.006918 - 0.399912*np.cos(B) + 0.070257*np.sin(B)
                           - 0.006758*np.cos(2*B) + 0.000907*np.sin(2*B)
                           - 0.002697*np.cos(3*B) + 0.00148*np.sin(3*B))
    
    delta = np.deg2rad(delta)
    ws = np.arccos(-np.tan(phi) * np.tan(delta))
    daylenght = (2/15) * np.rad2deg(ws)
    
    if year % 4 == 0:
        dayangle = np.deg2rad(360*day/366)
    else:
        dayangle = np.deg2rad(360*day/365)
    
    h0 = (24*3600*gsc/np.pi) * (1 + 0.033*np.cos(dayangle)) * (np.cos(phi)*np.cos(delta)*np.sin(ws) + 
                                                                     ws*np.sin(phi)*np.sin(delta))
    
    row['H0'] = h0
    row['Daylength'] = daylenght
    #print('Day: {} | Declination: {} | H0: {} | N: {}'.format(day, delta, h0/1e6, daylenght))
    return row

In [17]:
riz = data_dict['Rize']
ri = riz.reset_index()
ri = ri.apply(h0, axis=1)
ri.set_index('Date', inplace=True)

In [18]:
# Reset the index, and rename the column name to "Date"
# h0 is a function of columns Date and Latitude, we'll apply it here.
for key in data_dict:
    df = data_dict[key].reset_index()
    df = df.apply(h0, axis=1)
    data_dict[key] = df.set_index('Date')

In [19]:
print(len(data_dict))
print(data_dict['Rize'][:100])

46
           Station  Insolation  Daily Total  Temperature(avg)  Latitude  \
Date                                                                      
2011-01-01    Rize         0.1      32980.6          6.470833     41.04   
2011-01-02    Rize         2.4      84242.8          5.912500     41.04   
2011-01-03    Rize         4.8      99181.1          7.350000     41.04   
2011-01-04    Rize         3.4      74662.2         11.016667     41.04   
2011-01-05    Rize         1.0      43414.0          7.825000     41.04   
2011-01-06    Rize         0.7      48052.2          8.883333     41.04   
2011-01-07    Rize         1.0      73177.4          9.412500     41.04   
2011-01-08    Rize         0.6      49018.0          6.895833     41.04   
2011-01-09    Rize         0.0      40485.6          6.745833     41.04   
2011-01-10    Rize         0.7      66817.4          7.783333     41.04   
2011-01-11    Rize         2.4      97756.3          6.254167     41.04   
2011-01-12    Rize    

In [20]:
for k in data_dict:
    data_dict[k]['Daily Total'] *= (60 / 10**6)
    data_dict[k]['H0'] /= 10**6

with open('daily32-6cols.pkl', 'wb') as f:
    pickle.dump(data_dict, f)

In [2]:
with open('daily32-6cols.pkl', 'rb') as f:
    data_dict = pickle.load(f)

In [4]:
for key in data_dict:
    zero = np.where(data_dict[key]['H0']==0)
    if len(zero) > 0:
        print(f'{key} contains zeroes at {zero}')

Rize contains zeroes at (array([], dtype=int64),)
Artvin contains zeroes at (array([], dtype=int64),)
Kırklareli contains zeroes at (array([], dtype=int64),)
Bolu contains zeroes at (array([], dtype=int64),)
Kastamonu contains zeroes at (array([], dtype=int64),)
Tokat contains zeroes at (array([], dtype=int64),)
Gümüşhane contains zeroes at (array([], dtype=int64),)
Kars contains zeroes at (array([], dtype=int64),)
Ağrı contains zeroes at (array([], dtype=int64),)
Çanakkale contains zeroes at (array([], dtype=int64),)
Bursa contains zeroes at (array([], dtype=int64),)
Gemerek contains zeroes at (array([], dtype=int64),)
Van Bölge contains zeroes at (array([], dtype=int64),)
Afyonkarahisar Bölge contains zeroes at (array([], dtype=int64),)
Aksaray contains zeroes at (array([], dtype=int64),)
Malatya contains zeroes at (array([], dtype=int64),)
Akşehir contains zeroes at (array([], dtype=int64),)
Isparta contains zeroes at (array([], dtype=int64),)
Beyşehir contains zeroes at (array([], 

In [9]:
xs = data_dict['Aksaray'][['Latitude', 'Insolation', 'Temperature(avg)', 'Daylength', 'H0']]
ys = data_dict['Aksaray']['Daily Total']

assert len(xs) == len(ys)
print(xs[:10])
print(ys[:10])

            Latitude  Insolation  Temperature(avg)  Daylength            H0
Date                                                                       
2011-01-01   38.3705         1.7         -0.025000   9.373819  1.479024e+07
2011-01-02   38.3705         5.6          2.541667   9.384311  1.484324e+07
2011-01-03   38.3705         6.3          6.666667   9.395793  1.490124e+07
2011-01-04   38.3705         2.9          5.312500   9.408252  1.496422e+07
2011-01-05   38.3705         0.7          4.639130   9.421679  1.503214e+07
2011-01-06   38.3705         0.4          3.995833   9.436061  1.510500e+07
2011-01-07   38.3705         0.5          2.204167   9.451384  1.518275e+07
2011-01-08   38.3705         0.0          0.675000   9.467636  1.526536e+07
2011-01-09   38.3705         1.0          0.429167   9.484801  1.535281e+07
2011-01-10   38.3705         4.0          2.258333   9.502865  1.544505e+07
Date
2011-01-01     72927.6
2011-01-02    145874.7
2011-01-03    147197.8
2011-01-04    

In [10]:
print(xs.values[:10])
print(ys.values[:10])
print(xs.shape)
print(ys.shape)

[[  3.83705000e+01   1.70000000e+00  -2.50000000e-02   9.37381850e+00
    1.47902393e+07]
 [  3.83705000e+01   5.60000000e+00   2.54166667e+00   9.38431102e+00
    1.48432399e+07]
 [  3.83705000e+01   6.30000000e+00   6.66666667e+00   9.39579250e+00
    1.49012392e+07]
 [  3.83705000e+01   2.90000000e+00   5.31250000e+00   9.40825236e+00
    1.49642150e+07]
 [  3.83705000e+01   7.00000000e-01   4.63913043e+00   9.42167917e+00
    1.50321429e+07]
 [  3.83705000e+01   4.00000000e-01   3.99583333e+00   9.43606074e+00
    1.51049961e+07]
 [  3.83705000e+01   5.00000000e-01   2.20416667e+00   9.45138410e+00
    1.51827455e+07]
 [  3.83705000e+01   0.00000000e+00   6.75000000e-01   9.46763559e+00
    1.52653598e+07]
 [  3.83705000e+01   1.00000000e+00   4.29166667e-01   9.48480085e+00
    1.53528053e+07]
 [  3.83705000e+01   4.00000000e+00   2.25833333e+00   9.50286492e+00
    1.54450460e+07]]
[  72927.6  145874.7  147197.8  100350.9   42138.7   48120.9   62338.5
   30074.9   64647.1  125928

In [21]:
first = True
for key in data_dict:
    xs = data_dict[key][['Latitude', 'Insolation', 'Temperature(avg)', 'Daylength', 'H0']]
    ys = data_dict[key]['Daily Total']
    xs = xs.values
    ys = ys.values
    
    assert len(xs) == len(ys)
    if first:
        xall = xs
        yall = ys
        first = False
    else:
        xall = np.concatenate([xall, xs], axis=0)
        yall = np.concatenate([yall, ys], axis=0)
print('Xs have shape: {}, Ys have shape: {}'.format(xall.shape, yall.shape))

Xs have shape: (80149, 5), Ys have shape: (80149,)


In [22]:
print(xall[:5])
print(yall[:5])

[[  4.10400000e+01   1.00000000e-01   6.47083333e+00   9.09995719e+00
    1.31361933e+07]
 [  4.10400000e+01   2.40000000e+00   5.91250000e+00   9.11165032e+00
    1.31894429e+07]
 [  4.10400000e+01   4.80000000e+00   7.35000000e+00   9.12444399e+00
    1.32477425e+07]
 [  4.10400000e+01   3.40000000e+00   1.10166667e+01   9.13832599e+00
    1.33110734e+07]
 [  4.10400000e+01   1.00000000e+00   7.82500000e+00   9.15328316e+00
    1.33794151e+07]]
[ 32980.6  84242.8  99181.1  74662.2  43414. ]


In [23]:
print(data_dict['Rize'].head())

           Station  Insolation  Daily Total  Temperature(avg)  Latitude  \
Date                                                                      
2011-01-01    Rize         0.1      32980.6          6.470833     41.04   
2011-01-02    Rize         2.4      84242.8          5.912500     41.04   
2011-01-03    Rize         4.8      99181.1          7.350000     41.04   
2011-01-04    Rize         3.4      74662.2         11.016667     41.04   
2011-01-05    Rize         1.0      43414.0          7.825000     41.04   

                      H0  Daylength  
Date                                 
2011-01-01  1.313619e+07   9.099957  
2011-01-02  1.318944e+07   9.111650  
2011-01-03  1.324774e+07   9.124444  
2011-01-04  1.331107e+07   9.138326  
2011-01-05  1.337942e+07   9.153283  


In [24]:
alldat = {'xs':xall, 'ys':yall}
with open('dict_of_arrays.pickle', 'wb') as f:
    pickle.dump(alldat, f)

In [4]:
with open('dict_of_arrays.pickle', 'rb') as f:
    aldat = pickle.load(f)
    
print(aldat['xs'].shape)
print(aldat['ys'].shape)

(80149, 5)
(80149,)


In [11]:
# Now what's left to do is to insert longitude into the DataFrames
# A dictionary of station_name_to_longitude would be useful

station_name_to_latitude = {'Rize':41.0400, 'Artvin':41.1752, 'Kırklareli':41.7382, 'Bolu':40.7329, 
                             'Kastamonu':41.3710, 'Tokat':40.3312, 'Gümüşhane':40.4598, 'Kars':40.6042,
                             'Ağrı':39.7253, 'Çanakkale':40.1410, 'Bursa':40.2308, 'Gemerek':39.1850,
                             'Van Bölge':38.4693, 'Afyonkarahisar Bölge':38.7380, 'Aksaray':38.3705,
                             'Malatya':38.3367, 'Akşehir':38.3688, 'Isparta':37.7848, 'Beyşehir':37.6777,
                             'Karaman':37.1932, 'Kilis':36.7085, 'Hakkari':37.5745, 'Şırnak':37.5209,
                             'Adana Bölge':37.0041, 'Ünye':41.1430, 'Suşehri':40.1623, 'Tortum':40.3013,
                             'Tercan':39.7769, 'Doğubeyazıt':39.5396, 'Burhaniye':39.4983, 'Divriği':39.3618,
                             'Kemalpaşa':38.4639, 'Kulu':39.0788, 'Boğazlıyan':39.1897, 'Solhan':38.9597,
                             'Malazgirt':39.1436, 'Menemen':38.6237, 'Palu':38.6907, 'Develi':38.3744,
                             'Ergani':38.2670, 'Göksun':38.0240, 'Ulukışla':37.5480, 'Bozova':37.3651,
                             'Elmalı':36.7372, 'Ceylanpınar Tigem':36.8406, 'Tarsus':36.8942}

lat_to_statname = {v: k for k, v in station_name_to_latitude.items()}

test_list = ['Ergani', 'Adana Bölge', 'Bursa', 'Ünye', 'Boğazlıyan', 'Malazgirt']
dev_list = ['Doğubeyazıt', 'Tercan', 'Suşehri', 'Beyşehir', 'Menemen', 'Bozova']

In [12]:
first = True
for key in data_dict:
    if (key in dev_list) or (key in test_list): continue
    xs = data_dict[key][['Latitude', 'Insolation', 'Temperature(avg)', 'Daylength', 'H0']]
    ys = data_dict[key]['Daily Total']
    xs = xs.values
    ys = ys.values
    
    assert len(xs) == len(ys)
    if first:
        xall = xs
        yall = ys
        first = False
    else:
        xall = np.concatenate([xall, xs], axis=0)
        yall = np.concatenate([yall, ys], axis=0)
    print('Station {} has been added to the training set'.format(key))
print('Xs have shape: {}, Ys have shape: {}'.format(xall.shape, yall.shape))

Station Rize has been added to the training set
Station Artvin has been added to the training set
Station Kırklareli has been added to the training set
Station Bolu has been added to the training set
Station Kastamonu has been added to the training set
Station Tokat has been added to the training set
Station Gümüşhane has been added to the training set
Station Kars has been added to the training set
Station Ağrı has been added to the training set
Station Çanakkale has been added to the training set
Station Gemerek has been added to the training set
Station Van Bölge has been added to the training set
Station Afyonkarahisar Bölge has been added to the training set
Station Aksaray has been added to the training set
Station Malatya has been added to the training set
Station Akşehir has been added to the training set
Station Isparta has been added to the training set
Station Karaman has been added to the training set
Station Kilis has been added to the training set
Station Hakkari has been

In [13]:
first = True
for key in data_dict:
    if key not in test_list: continue
    xs = data_dict[key][['Latitude', 'Insolation', 'Temperature(avg)', 'Daylength', 'H0']]
    ys = data_dict[key]['Daily Total']
    xs = xs.values
    ys = ys.values
    
    assert len(xs) == len(ys)
    if first:
        xall_test = xs
        yall_test = ys
        first = False
    else:
        xall_test = np.concatenate([xall_test, xs], axis=0)
        yall_test = np.concatenate([yall_test, ys], axis=0)
    print('Station {} has been added to the test set'.format(key))
print('Xs have shape: {}, Ys have shape: {}'.format(xall_test.shape, yall_test.shape))

Station Bursa has been added to the test set
Station Adana Bölge has been added to the test set
Station Ünye has been added to the test set
Station Boğazlıyan has been added to the test set
Station Malazgirt has been added to the test set
Station Ergani has been added to the test set
Xs have shape: (10601, 5), Ys have shape: (10601,)


In [14]:
first = True
for key in data_dict:
    if key not in dev_list: continue
    xs = data_dict[key][['Latitude', 'Insolation', 'Temperature(avg)', 'Daylength', 'H0']]
    ys = data_dict[key]['Daily Total']
    xs = xs.values
    ys = ys.values
    
    assert len(xs) == len(ys)
    if first:
        xall_dev = xs
        yall_dev = ys
        first = False
    else:
        xall_dev = np.concatenate([xall_dev, xs], axis=0)
        yall_dev = np.concatenate([yall_dev, ys], axis=0)
    print('Station {} has been added to the test set'.format(key))
print('Xs have shape: {}, Ys have shape: {}'.format(xall_dev.shape, yall_dev.shape))

Station Beyşehir has been added to the test set
Station Suşehri has been added to the test set
Station Tercan has been added to the test set
Station Doğubeyazıt has been added to the test set
Station Menemen has been added to the test set
Station Bozova has been added to the test set
Xs have shape: (9019, 5), Ys have shape: (9019,)


In [23]:
alldat2 = {'x_train':xall, 'y_train':yall, 'x_dev':xall_dev, 'y_dev':yall_dev, 'x_test':xall_test, 'y_test':yall_test}
with open('splitted_alldat.pkl', 'wb') as f:
    pickle.dump(alldat2, f)
    
with open('splitted_alldat.pkl', 'rb') as f:
    alldat = pickle.load(f)
    
print(alldat2['x_train'][:10])
print(alldat['x_train'][:10])

[[  4.10400000e+01   1.00000000e-01   6.47083333e+00   9.09995719e+00
    1.31361933e+07]
 [  4.10400000e+01   2.40000000e+00   5.91250000e+00   9.11165032e+00
    1.31894429e+07]
 [  4.10400000e+01   4.80000000e+00   7.35000000e+00   9.12444399e+00
    1.32477425e+07]
 [  4.10400000e+01   3.40000000e+00   1.10166667e+01   9.13832599e+00
    1.33110734e+07]
 [  4.10400000e+01   1.00000000e+00   7.82500000e+00   9.15328316e+00
    1.33794151e+07]
 [  4.10400000e+01   7.00000000e-01   8.88333333e+00   9.16930144e+00
    1.34527448e+07]
 [  4.10400000e+01   1.00000000e+00   9.41250000e+00   9.18636592e+00
    1.35310380e+07]
 [  4.10400000e+01   6.00000000e-01   6.89583333e+00   9.20446087e+00
    1.36142678e+07]
 [  4.10400000e+01   0.00000000e+00   6.74583333e+00   9.22356980e+00
    1.37024053e+07]
 [  4.10400000e+01   7.00000000e-01   7.78333333e+00   9.24367552e+00
    1.37954196e+07]]
[[  4.10400000e+01   1.00000000e-01   6.47083333e+00   9.09995719e+00
    1.31361933e+07]
 [  4.104